In [1]:
!pip install swig
!pip install gymnasium[box2d] moviepy

import gymnasium as gym
import numpy as np
import tensorflow as tf
import moviepy.editor as mpy
import os

env = gym.make('CartPole-v1', render_mode="rgb_array_list")

max_steps = 50_000
step = 0
lr = 0.005
gamma = 0.9999

model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(4,)),
    tf.keras.layers.Dense(env.action_space.n, activation='softmax')
])

optimizer = tf.keras.optimizers.Adam(learning_rate=lr)

# entrenamiento
while step <= max_steps:
    obs, _ = env.reset()
    done = False
    Actions, States, Rewards = [], [], []

    while not done:
        obs_tensor = tf.convert_to_tensor([obs], dtype=tf.float32)
        probs = model(obs_tensor)
        action = tf.random.categorical(tf.math.log(probs), 1)[0, 0].numpy()

        obs_, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        Actions.append(action)
        States.append(obs)
        Rewards.append(reward)

        obs = obs_

        step += 1

    # calculo retornos descontados
    DiscountedReturns = []
    for t in range(len(Rewards)):
        G = 0.0
        for k, r in enumerate(Rewards[t:]):
            G += (gamma**k) * r
        DiscountedReturns.append(G)

    # actualiz. modelo
    for state, action, G in zip(States, Actions, DiscountedReturns):
        with tf.GradientTape() as tape:
            state_tensor = tf.convert_to_tensor([state], dtype=tf.float32)
            probs = model(state_tensor, training=True)
            action_prob = tf.gather(probs[0], action)
            log_prob = tf.math.log(action_prob + 1e-8)  #
            loss = -log_prob * G

        grads = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))

# test
print("\nGenerando video...")

video_folder = "./cartpole_videos/"
os.makedirs(video_folder, exist_ok=True)

for episode in range(1):
    obs, _ = env.reset()
    done = False
    frames = []
    Rewards = []

    while not done:
        obs_tensor = tf.convert_to_tensor([obs], dtype=tf.float32)
        probs = model(obs_tensor)
        action = tf.random.categorical(tf.math.log(probs), 1)[0, 0].numpy()

        obs_, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        Rewards.append(reward)
        obs = obs_

    frames = env.render()

# guarda video
video_path = os.path.join(video_folder, "cartpole_run.mp4")
clip = mpy.ImageSequenceClip(frames, fps=30)
clip.write_videofile(video_path)

print(f"Video guardado en {video_path}")

env.close()


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.4/374.4 kB 10.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 42.0 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for box2d-py
  Running setup.py clean for box2d-py
Failed to build box2d-py
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (box2d-py)


/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':

  super().__init__(activity_regularizer=activity_regularizer, **kwargs)




Generando video...
Moviepy - Building video ./cartpole_videos/cartpole_run.mp4.
Moviepy - Writing video ./cartpole_videos/cartpole_run.mp4



Moviepy - Done !
Moviepy - video ready ./cartpole_videos/cartpole_run.mp4
Video guardado en ./cartpole_videos/cartpole_run.mp4
